<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drexelwireless/ece512-courseware/blob/main/homework/hw01/hw01.ipynb)

In [ ]:
%pip install -q otter-grader

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook()

# ECE 512 Wireless Communications — Homework 1: Cellular Geometry, Frequency Reuse, and Friis

This homework builds on the figures of the
[Week 1 lecture notebook](https://colab.research.google.com/github/drexelwireless/ece512-courseware/blob/main/notebooks/lectures/week01-cellular-geometry.ipynb),
drawn for **your own network**: your cluster pattern, your link, your mobile driving through your cells, and the
interference it sees. All the code is written for you and hidden; you **run the cells, look at the figures, and type
what you read** — a number, a letter, or `True`/`False`. This homework is about wireless, not programming.

**How it works**
* Work in Google Colab. Run the cells from top to bottom (*Runtime → Run all* runs everything).
* Cells with a title and a ▶ button are figure cells: just run them. (In Colab their code is hidden; you never need it.
  Outside Colab you'll see the code; you can ignore it.)
* In each answer cell, replace the `...` with your answer, run it, then run the `grader.check(...)` cell below it.
  The check only tests the *format*; your values are graded afterwards against **your** figures.
* **Your numbers are personal** — they come from your Drexel user ID, so they differ from your classmates'.
* Follow the course AI policy — the checks are quick; the learning is the point.

### The only Python you need here

* **Run a cell:** click it and press **Shift + Enter** (or the ▶ button).
* **Answer:** replace `...` with what you read, e.g. `q2a = 7`, `q4a = -49.0`, `q2c = "a"` (letters in quotes),
  `q6c = True` (capital `T`/`F`), `q6b = (12, 3)` (two numbers in parentheses).
* **Arithmetic works too:** `q4b = 55.02 - 49.00`. Powers use `**`, so $x^2$ is `x**2`; $\sqrt{x}$ is `np.sqrt(x)`.
* **If you see an error**, read its **last line**. `NameError: name '…' is not defined` means a cell above has not been
  run: use *Runtime → Run all*. `SyntaxError` is usually a missing quote or bracket in your answer.

In [ ]:
# @title Setup — run this first (installs the course tools)
COURSEWARE_URL = "git+https://github.com/drexelwireless/ece512-courseware"
TESTS_URL = "https://raw.githubusercontent.com/drexelwireless/ece512-courseware/main/homework/hw01/hw01-tests.zip"
import importlib.util, io, os, urllib.request, zipfile
if importlib.util.find_spec("ece512") is None:
    get_ipython().run_line_magic("pip", f"install -q {COURSEWARE_URL}")
if not os.path.isdir("tests"):
    try:
        zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(TESTS_URL, timeout=30).read())).extractall(".")
    except Exception as err:   # offline, or the homework is not published yet
        print(f"Could not download the format checks ({err}). Everything else works; "
              "grader.check(...) cells will report errors until the checks are available.")

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider
from ece512 import cells, propagation, pathloss as pl, animation
from ece512.hexgrid import draw_cell, fill_cell, cell_axes
from ece512.student import student_rng
LETTERS = "ABCDEFGHIJKLM"
GROUP_COLORS = plt.cm.tab10(np.arange(10))
print("Setup done.")

### Your ID
Type your Drexel user ID between the quotes (e.g. `"abc123"`) and run the cell, then run the next cell.

In [ ]:
STUDENT_ID = ...

In [ ]:
# @title Your parameters — run this (it reads your ID)
rng = student_rng(STUDENT_ID, "hw01")
_R = float(rng.choice([200, 300, 400, 500, 750, 1000]))
_i, _j = [(1, 1), (2, 0), (2, 1)][int(rng.integers(3))]
_start = 2.5 * _R * np.exp(1j * rng.uniform(0, 2 * np.pi))
_end = -_start * np.exp(1j * rng.uniform(-0.6, 0.6))
params = {
    "R_m": _R,                                   # cell radius R (m)
    "i": _i, "j": _j,                            # your network's cluster shift parameters
    "S_channels": int(rng.choice([280, 336, 395, 420, 504, 588])),   # total duplex channels S
    "sir_req_db": float(rng.choice([11, 12, 13, 15, 17, 18, 19])),   # required worst-case SIR (dB)
    "carrier_mhz": float(rng.choice([700, 850, 1900, 2100, 2600, 3500])),
    "ptx_dbm": float(rng.choice([30, 33, 36, 40, 43, 46])),
    "gtx_dbi": float(rng.choice([8, 10, 12, 15, 17])),
    "grx_dbi": float(rng.choice([0, 2])),
    "lsys_db": float(rng.choice([1, 2, 3])),
    "hysteresis_db": float(rng.choice([3, 4, 5, 6])),              # handoff hysteresis h (dB)
    "sigma_db": float(rng.choice([1.5, 2.0, 2.5, 3.0])),            # measurement noise (dB)
    "start": complex(np.round(_start, 1)),       # trajectory start (m)
    "end": complex(np.round(_end, 1)),           # trajectory end (m)
    "n_positions": 400,
    "noise_seed": int(rng.integers(2**31)),
}

p = params
print(f"""Your network and link
  cell radius R .............. {p['R_m']:g} m
  cluster shift (i, j) ....... ({p['i']}, {p['j']})
  total channels S ........... {p['S_channels']}
  required worst-case SIR .... {p['sir_req_db']:g} dB
  carrier frequency .......... {p['carrier_mhz']:g} MHz
  transmit power ............. {p['ptx_dbm']:g} dBm
  antenna gains (TX, RX) ..... {p['gtx_dbi']:g} dBi, {p['grx_dbi']:g} dBi
  system loss ................ {p['lsys_db']:g} dB
  handoff hysteresis h ....... {p['hysteresis_db']:g} dB
  measurement noise .......... {p['sigma_db']:g} dB""")

## Part A — Your cluster pattern, interference and capacity

From the lecture: a cluster of $N = i^2 + ij + j^2$ cells shares the spectrum, so each letter (A, B, C, …) is a different
set of channels. Cells with the **same letter** reuse the same channels; the closest ones are the **co-channel cells**, a
distance $D = R\sqrt{3N}$ apart. Figure A1 is the lecture's reuse map for **your** $(i, j)$: the dark cells all use
channel set **A**, and the arrow joins two of them.

In [ ]:
# @title Figure A1 — your frequency-reuse pattern
R, i, j = params["R_m"], params["i"], params["j"]
fig, ax = plt.subplots(figsize=(7.5, 7.5))
cell_axes(ax)
offsets = np.concatenate([[0j], cells.cochannel_offsets(i, j, R)])
for off in offsets:
    for g, c in enumerate(cells.cluster_centers(off, i, j, R)):
        fill_cell(ax, c, R, GROUP_COLORS[g % 10], alpha=0.75 if g == 0 else 0.18)
        draw_cell(ax, c, R, label=LETTERS[g], color="0.45")
d1 = offsets[1]
ax.annotate("", xy=(d1.real, d1.imag), xytext=(0, 0), arrowprops=dict(arrowstyle="<->", color="k", lw=2))
ax.text(d1.real / 2, d1.imag / 2, "  D", fontsize=14, fontweight="bold")
lim = abs(d1) + 2.2 * R
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.grid(alpha=0.25)
ax.set_title(f"Your network: i = {i}, j = {j}, cell radius R = {R:g} m\n"
             "dark = cells that reuse channel set A; arrow = reuse distance D")
plt.show()

**Question 1 — read your cluster (2 pts).**

* `q1_N` — your cluster size $N$: count the **different letters** in Figure A1 (one cluster uses each letter once).
  Check it with the formula $N = i^2 + ij + j^2$ and your $i, j$.
* `q1_D_m` — your reuse distance $D = R\sqrt{3N}$ in meters, for your $R$. For example, with $R = 300$ m and $N = 3$ you
  would type `q1_D_m = 300 * np.sqrt(3 * 3)`. Check that it matches the length of the arrow (the axes are in meters).

In [ ]:
q1_N = ...
q1_D_m = ...

In [ ]:
grader.check("q1")

How large must $N$ be? The lecture's estimate puts all six co-channel interferers at distance $D$:
$\Lambda \approx Q^\beta/6$ with $Q = D/R = \sqrt{3N}$. The **worst case** is a mobile at a **corner** of its cell, which
is only about $D - R$ from some interferers. Figure A2 shows both for $\beta = 4$, your requirement, and — on the right —
how many channels each cell gets, $k = \lfloor S/N \rfloor$, from your $S$ channels. Bigger clusters mean less
interference but fewer channels per cell.

In [ ]:
# @title Figure A2 — interference vs. capacity for each cluster size
S = params["S_channels"]
cluster_table = []
for (a, b) in [(1, 0), (1, 1), (2, 0), (2, 1), (3, 0), (2, 2), (3, 1)]:
    n = cells.cluster_size(a, b)
    cluster_table.append(dict(N=n, approx_db=10 * np.log10(np.sqrt(3 * n) ** 4 / 6),
                              worst_db=10 * np.log10(pl.worst_case_sir_geometric(a, b, 1.0, 4)), k=S // n))
Ns = [r["N"] for r in cluster_table]
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw=dict(width_ratios=[3, 2]))
ax.plot(Ns, [r["approx_db"] for r in cluster_table], "o--", color="tab:gray", label="estimate Q^4/6 (all interferers at D)")
ax.plot(Ns, [r["worst_db"] for r in cluster_table], "s-", color="tab:blue", lw=2, label="exact worst case (mobile at a corner)")
for r in cluster_table:
    ax.annotate(f"{r['worst_db']:.1f}", (r["N"], r["worst_db"]), xytext=(0, -16), textcoords="offset points",
                ha="center", color="tab:blue", fontsize=9)
ax.axhline(params["sir_req_db"], color="tab:red", lw=1.5, ls=":", label=f"your requirement: {params['sir_req_db']:g} dB")
ax.margins(y=0.12); ax.set_xticks(Ns); ax.set_xlabel("cluster size N"); ax.set_ylabel("SIR (dB), beta = 4")
ax.set_title("Interference: bigger clusters are better"); ax.grid(alpha=0.3); ax.legend(loc="lower right", fontsize=9)
bars = ax2.bar([str(n) for n in Ns], [r["k"] for r in cluster_table], color="tab:green", alpha=0.7)
ax2.bar_label(bars, fontsize=9)
ax2.set_xlabel("cluster size N"); ax2.set_ylabel("channels per cell k")
ax2.set_title(f"Capacity: your S = {S} channels shared by N cells"); ax2.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

**Question 2 — choose your cluster size (3 pts).** Use Figure A2.

* `q2a` — the **smallest** $N$ whose **exact worst case** (solid blue, value printed under each point) is at or above your
  requirement (red dotted line).
* `q2b` — the number of channels per cell $k$ for that $N$ (green bar on the right).
* `q2c` — why is the estimate $Q^4/6$ (gray) always **above** the exact worst case? One letter:
  * `"a"` — a mobile at a cell corner is only about $D - R$ from the nearest co-channel base stations, not $D$
  * `"b"` — the estimate counts too many interferers
  * `"c"` — the estimate assumes free space ($\beta = 2$)
  * `"d"` — at a corner the mobile is farther than $R$ from its own base station

In [ ]:
q2a = ...
q2b = ...
q2c = ...

In [ ]:
grader.check("q2")

## Part B — Your link: Friis free-space received power

$$P_{\mathrm{RX|dBm}}(d) = P_{\mathrm{TX|dBm}} + G_{\mathrm{TX|dBi}} + G_{\mathrm{RX|dBi}} - L_{\mathrm{sys|dB}}
- 20\log_{10}\!\frac{4\pi d}{\lambda}$$
Figure B1 is the lecture's Friis plot for **your** link (your power, gains, loss and carrier), with the received power
marked at your cell edge $d = R$ and at $d = 2R$.

In [ ]:
# @title Figure B1 — received power on your link
f_hz = params["carrier_mhz"] * 1e6
def your_link_dbm(d_m, f=f_hz):
    return propagation.friis_received_power_dbm(params["ptx_dbm"], d_m, f, params["gtx_dbi"],
                                                params["grx_dbi"], params["lsys_db"])
R = params["R_m"]
d = np.logspace(np.log10(R / 10), np.log10(20 * R), 300)
link = {"R": float(your_link_dbm(R)), "2R": float(your_link_dbm(2 * R))}
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.semilogx(d, your_link_dbm(d), lw=2, label=f"your link at {params['carrier_mhz']:g} MHz")
for name, dd, off in [("R", R, (12, 10)), ("2R", 2 * R, (12, 10))]:
    ax.plot(dd, link[name], "o", color="tab:red", ms=8)
    ax.annotate(f"d = {name} = {dd:g} m\nP_RX = {link[name]:.2f} dBm", (dd, link[name]), xytext=off,
                textcoords="offset points", fontsize=10, color="tab:red")
ax.set_xlabel("distance d (m)"); ax.set_ylabel("received power P_RX (dBm)")
ax.grid(True, which="both", alpha=0.3); ax.legend(); plt.show()

**Question 3 — what went wrong? (2 pts).** A teammate made Figure B2 with their own Friis calculation (orange) for a
30 dBm transmitter at 900 MHz with unity-gain antennas. The correct curve is blue. Their received power falls only
**10 dB per decade** of distance instead of 20. Which mistake explains it? Run Figure B2, then answer with one letter:

* `"a"` — they wrote the path loss as $10\log_{10}(4\pi d/\lambda)$ instead of $20\log_{10}(4\pi d/\lambda)$, so their power
  falls as $1/d$ instead of $1/d^2$
* `"b"` — they used the wrong speed of light to compute the wavelength
* `"c"` — they forgot to add the antenna gains
* `"d"` — they entered the distance in kilometers instead of meters

*Hint:* three of these mistakes move the whole curve up or down by a fixed number of dB; only one changes its **slope**.

In [ ]:
q3 = ...

In [ ]:
grader.check("q3")

In [ ]:
# @title Figure B2 — a teammate's Friis curve
d = np.logspace(1, 4, 100)
lam = 299792458.0 / 900e6
correct = 30 - 20 * np.log10(4 * np.pi * d / lam)
teammate = 30 - 10 * np.log10(4 * np.pi * d / lam)
fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.semilogx(d, correct, lw=2, color="tab:blue", label="correct: falls 20 dB per decade")
ax.semilogx(d, teammate, lw=2, ls="--", color="tab:orange", label="teammate: falls 10 dB per decade")
ax.set_xlabel("distance d (m)"); ax.set_ylabel("received power (dBm)")
ax.set_title("30 dBm, 900 MHz, unity-gain antennas"); ax.grid(True, which="both", alpha=0.3); ax.legend()
plt.show()

**Question 4 — read your link (2 pts).** Use the red labels in Figure B1.

* `q4a` — the received power (dBm) at your cell edge, $d = R$.
* `q4b` — how many dB **weaker** the signal is at $d = 2R$ than at $d = R$ (a positive number; you may type the
  subtraction, e.g. `q4b = 55.02 - 49.00`). Compare it with the lecture's 20 dB per decade.

In [ ]:
q4a = ...
q4b = ...

In [ ]:
grader.check("q4")

*Explore (not graded):* move the slider to compare your link with another carrier frequency. Does the gap between the
curves depend on distance? How much does each doubling of the frequency cost?

In [ ]:
# @title Explore — carrier frequency
def explore_frequency(other_mhz=3500.0):
    d = np.logspace(np.log10(params["R_m"] / 10), np.log10(20 * params["R_m"]), 200)
    fig, ax = plt.subplots(figsize=(8.5, 4))
    ax.semilogx(d, your_link_dbm(d), lw=2, label=f"your carrier, {params['carrier_mhz']:g} MHz")
    ax.semilogx(d, your_link_dbm(d, other_mhz * 1e6), lw=2, ls="--", label=f"{other_mhz:g} MHz")
    gap = float(your_link_dbm(1.0) - your_link_dbm(1.0, other_mhz * 1e6))
    ax.set_title(f"gap = {gap:.1f} dB at every distance")
    ax.set_xlabel("distance d (m)"); ax.set_ylabel("P_RX (dBm)"); ax.grid(True, which="both", alpha=0.3); ax.legend()
    plt.show()

interact(explore_frequency, other_mhz=FloatSlider(value=3500, min=500, max=28000, step=100));

## Part C — Driving through your network: serving cell and handoff

Your mobile drives in a straight line through your network (Figure C1, red line; the dot is the start). At every
point it measures the power from all base stations (Friis, as in Part B) and is **served** by the strongest one. Real
measurements fluctuate (next week: shadowing), so we also run a **noisy** version with random measurement errors.

Real networks do not hand off the instant a neighbor becomes stronger: they wait until the neighbor is **more than $h$
dB stronger** (the handoff **hysteresis**), to avoid "ping-pong" handoffs. With $h = 0$ the mobile is always served by
the nearest base station, as in the lecture.

In [ ]:
# @title Figure C1 — your network and your drive
def hex_layout(i, j, R, radius):
    u1, u2 = cells.cochannel_offsets(i, j, R)[:2]
    reps = cells.cluster_centers(0j, i, j, R)
    K = int(np.ceil(radius / cells.reuse_distance(i, j, R))) + 2
    bs, group = [], []
    for m in range(-K, K + 1):
        for n in range(-K, K + 1):
            for g, rep in enumerate(reps):
                c = rep + m * u1 + n * u2
                if abs(c) <= radius:
                    bs.append(c); group.append(g)
    return np.array(bs), np.array(group)

def serving_with_hysteresis(rx, h_db):
    serving = np.empty(rx.shape[0], dtype=int)
    current = int(np.argmax(rx[0]))
    for t in range(rx.shape[0]):
        best = int(np.argmax(rx[t]))
        if rx[t, best] > rx[t, current] + h_db:
            current = best
        serving[t] = current
    return serving

R, i, j = params["R_m"], params["i"], params["j"]
bs_xy, bs_group = hex_layout(i, j, R, 9 * R)
positions = np.linspace(params["start"], params["end"], params["n_positions"])
traveled = np.abs(positions - positions[0])
rx_dbm = your_link_dbm(np.abs(positions[:, None] - bs_xy[None, :]))
rx_meas_dbm = rx_dbm + np.random.default_rng(params["noise_seed"]).normal(0, params["sigma_db"], rx_dbm.shape)
h = params["hysteresis_db"]
serving_0 = serving_with_hysteresis(rx_dbm, 0.0)
serving_h = serving_with_hysteresis(rx_dbm, h)

ax = cell_axes(figsize=(7.5, 7.5))
for c, g in zip(bs_xy, bs_group):
    if abs(c) < 4.5 * R:
        fill_cell(ax, c, R, GROUP_COLORS[g % 10], alpha=0.25)
        draw_cell(ax, c, R, label=LETTERS[g], color="0.5")
ax.plot(positions.real, positions.imag, "r-", lw=2.5)
ax.plot(positions[0].real, positions[0].imag, "ro", ms=9)
ax.annotate("", xy=(positions[-1].real, positions[-1].imag), xytext=(positions[-12].real, positions[-12].imag),
            arrowprops=dict(arrowstyle="->", color="r", lw=2.5))
ax.set_xlim(-4.5 * R, 4.5 * R); ax.set_ylim(-4.5 * R, 4.5 * R)
ax.set_title(f"Your drive: {traveled[-1]:.0f} m (dot = start)")
plt.show()

In [ ]:
# @title Optional movie — tick the box (in Colab) or set SHOW_MOVIE = True, then run
SHOW_MOVIE = False  # @param {type:"boolean"}
if SHOW_MOVIE:
    frames = np.arange(0, len(positions), 8)
    def draw_frame(ax, k):
        t = frames[k]
        cell_axes(ax)
        for b, (c, g) in enumerate(zip(bs_xy, bs_group)):
            if abs(c) < 4.5 * R:
                fill_cell(ax, c, R, GROUP_COLORS[g % 10], alpha=0.85 if b == serving_h[t] else 0.15)
                draw_cell(ax, c, R, label=LETTERS[g], color="0.5")
        ax.plot(positions[:t + 1].real, positions[:t + 1].imag, "r-", lw=1.5)
        ax.plot(positions[t].real, positions[t].imag, "ro", ms=8)
        ax.set_xlim(-4.5 * R, 4.5 * R); ax.set_ylim(-4.5 * R, 4.5 * R)
        ax.set_title(f"{traveled[t]:.0f} m: served by the highlighted {LETTERS[bs_group[serving_h[t]]]} cell (h = {h:g} dB)")
    display(animation.show(animation.animate(draw_frame, len(frames), figsize=(6.5, 6.5), interval_ms=120)))
else:
    print("Movie skipped.")

**Question 5 — predict first (1 pt; any answer earns the point).** *Before* running Figure C2: with your hysteresis $h$
and no noise, where will the **first** handoff happen compared with $h = 0$ (nearest base station)?

* `"a"` — exactly at the hexagon boundary, same as $h = 0$
* `"b"` — before the mobile reaches the boundary
* `"c"` — after the mobile has crossed the boundary into the new cell
* `"d"` — never — hysteresis prevents handoffs on a straight path

In [ ]:
q5 = ...

In [ ]:
grader.check("q5")

In [ ]:
# @title Figure C2 — power from the serving base station along your drive
serving_meas_0 = serving_with_hysteresis(rx_meas_dbm, 0.0)
serving_meas_h = serving_with_hysteresis(rx_meas_dbm, h)
runs = [("no noise, h = 0 dB", rx_dbm, serving_0), (f"no noise, h = {h:g} dB", rx_dbm, serving_h),
        ("noisy, h = 0 dB", rx_meas_dbm, serving_meas_0), (f"noisy, h = {h:g} dB", rx_meas_dbm, serving_meas_h)]
drive = {}
fig, axes = plt.subplots(4, 1, figsize=(10.5, 12), sharex=True)
for ax, (name, rx_, s_) in zip(axes, runs):
    edges = np.flatnonzero(np.diff(s_)) + 1
    ax.plot(traveled, rx_[np.arange(len(s_)), s_], lw=1.2, color="k")
    for a, b in zip(np.r_[0, edges], np.r_[edges, len(s_)]):
        ax.axvspan(traveled[a], traveled[b - 1], alpha=0.25, color=GROUP_COLORS[bs_group[s_[a]] % 10])
    for e in edges:
        ax.axvline(traveled[e], color="k", lw=0.6, alpha=0.5)
    first = float(traveled[edges[0]])
    drive[name] = dict(n=len(edges), first=first)
    ax.annotate(f"first handoff at {first:.1f} m", (first, ax.get_ylim()[1]), xytext=(6, -16),
                textcoords="offset points", color="tab:red", fontsize=10, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="tab:red", alpha=0.95), zorder=6)
    ax.axvline(first, color="tab:red", lw=1.5)
    ax.set_title(f"{name}:  {len(edges)} handoffs", loc="left", fontweight="bold")
    ax.set_ylabel("P_RX from serving\nbase station (dBm)"); ax.grid(alpha=0.3)
axes[-1].set_xlabel("distance traveled (m)   (shading color = letter of the serving cell)")
plt.tight_layout(); plt.show()
print("Summary of Figure C2")
for name, v in drive.items():
    print(f"  {name:>20}:  {v['n']:3d} handoffs, first handoff at {v['first']:8.1f} m")

**Question 6 — read your drive (4 pts).** Use Figure C2 and the summary printed under it.

* `q6a` — how many meters **further** did the mobile travel before its first handoff with your hysteresis than with
  $h = 0$ (both **no noise**)? Type the subtraction, e.g. `q6a = 655.2 - 412.3`.
* `q6b` — the number of handoffs in the two **noisy** runs, as `(without hysteresis, with hysteresis)`, e.g. `(14, 3)`.
* `q6c` — `True` if, with no noise, hysteresis made **fewer** handoffs than $h = 0$ (it skipped a cell the drive only
  clipped), otherwise `False`.
* `q6d` — the mobile is served by the **closest** base station, as if each cell were a circle around its base station,
  yet the cells are drawn as hexagons. With no noise and $h = 0$, the handoffs happen:
  * `"a"` — exactly where the drive crosses a hexagon edge: the points closest to a base station form its hexagon
  * `"b"` — inside the hexagon, wherever two circles of radius $R$ overlap
  * `"c"` — only at the corners of the hexagons
  * `"d"` — at random places, because circles leave gaps between the cells

In [ ]:
q6a = ...
q6b = ...
q6c = ...
q6d = ...

In [ ]:
grader.check("q6")

*Explore (not graded):* move the slider to see how the number of handoffs in the **noisy** run depends on $h$. Which
$h$ would you choose, and what does a large $h$ cost?

In [ ]:
# @title Explore — handoff hysteresis
def explore_hysteresis(h_db=3.0):
    s_ = serving_with_hysteresis(rx_meas_dbm, h_db)
    edges = np.flatnonzero(np.diff(s_)) + 1
    fig, ax = plt.subplots(figsize=(10.5, 3))
    ax.plot(traveled, rx_meas_dbm[np.arange(len(s_)), s_], lw=1, color="k")
    for a, b in zip(np.r_[0, edges], np.r_[edges, len(s_)]):
        ax.axvspan(traveled[a], traveled[b - 1], alpha=0.25, color=GROUP_COLORS[bs_group[s_[a]] % 10])
    ax.set_title(f"noisy, h = {h_db:g} dB: {len(edges)} handoffs", loc="left", fontweight="bold")
    ax.set_xlabel("distance traveled (m)"); ax.set_ylabel("P_RX (dBm)"); ax.grid(alpha=0.3); plt.show()

interact(explore_hysteresis, h_db=FloatSlider(value=params["hysteresis_db"], min=0, max=12, step=0.5));

## Part D — Co-channel interference along your drive

At each point, the interference comes from every **other** base station with the **same letter** as the serving cell.
The SIR is the serving power divided by the **total** interference power. Powers from different base stations add as
**powers** (milliwatts), not as dB values: two interferers at −90 dBm each give −87 dBm in total (twice the power is
+3 dB), not −180 dBm.

In [ ]:
# @title Figure D1 — a teammate's SIR calculation
def sir_db(rx, serving, power_sum):
    out = np.empty(rx.shape[0])
    for t, s in enumerate(serving):
        co = (bs_group == bs_group[s]) & (np.arange(rx.shape[1]) != s)
        out[t] = rx[t, s] - power_sum(rx[t, co])
    return out

def total_dbm(p_dbm):                      # correct: convert to mW, add, convert back
    return 10 * np.log10(np.sum(10 ** (p_dbm / 10)))

sir_0 = sir_db(rx_dbm, serving_0, total_dbm)
sir_h = sir_db(rx_dbm, serving_h, total_dbm)
teammate = sir_db(rx_dbm, serving_0, np.sum)
fig, ax = plt.subplots(figsize=(10.5, 3.8))
ax.plot(traveled, teammate, color="tab:orange", lw=2, ls="--", label="teammate's SIR")
ax.plot(traveled, sir_0, color="tab:blue", lw=2, label="correct SIR")
ax.set_xlabel("distance traveled (m)"); ax.set_ylabel("SIR (dB)"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Same drive, h = 0: the teammate's SIR is hundreds of dB or more")
plt.show()

**Question 7 — what went wrong? (2 pts).** The teammate's SIR in Figure D1 (orange) is hundreds of dB or more, absurdly far above
anything Figure A2 suggests. Which mistake explains it? One letter:

* `"a"` — they added the interference powers **in dBm** (e.g. −90 + −90 = −180) instead of converting to milliwatts,
  adding, and converting back
* `"b"` — they used path-loss exponent $\beta = 4$ instead of the free-space $\beta = 2$
* `"c"` — they swapped the serving and the interfering base stations
* `"d"` — they forgot the handoff hysteresis

In [ ]:
q7 = ...

In [ ]:
grader.check("q7")

In [ ]:
# @title Figure D2 — the SIR along your drive
fig, ax = plt.subplots(figsize=(10.5, 4.4))
ax.plot(traveled, sir_0, color="tab:blue", lw=2, label="nearest base station (h = 0)")
ax.plot(traveled, sir_h, color="tab:purple", lw=1.5, ls="--", label=f"hysteresis h = {h:g} dB")
for y, col in [(sir_0, "tab:blue"), (sir_h, "tab:purple")]:
    k = int(np.argmin(y))
    ax.plot(traveled[k], y[k], "o", color=col, ms=9, zorder=5)
ax.text(0.99, 0.97, f"minimum SIR (dots)\nh = 0:      {sir_0.min():6.2f} dB\nh = {h:g} dB:  {sir_h.min():6.2f} dB",
        transform=ax.transAxes, ha="right", va="top", fontsize=11, fontweight="bold", family="monospace",
        bbox=dict(boxstyle="round", fc="white", ec="0.6"))
ax.axhline(params["sir_req_db"], color="tab:red", ls=":", lw=1.5, label=f"your requirement ({params['sir_req_db']:g} dB)")
ax.axhline(0, color="k", lw=1, label="0 dB: interference = signal")
ax.set_xlabel("distance traveled (m)"); ax.set_ylabel("SIR (dB)"); ax.grid(alpha=0.3); ax.legend(fontsize=9, loc="upper left")
ax.set_title("Free space (beta = 2) along your drive")
plt.show()
print(f"Minimum SIR: {sir_0.min():.2f} dB with h = 0,  {sir_h.min():.2f} dB with h = {h:g} dB;  "
      f"your requirement: {params['sir_req_db']:g} dB")

**Question 8 — read your SIR (3 pts).** Use Figure D2 and the line printed under it. Note that the channel here is
**free space** ($\beta = 2$), while your Part A requirement assumed $\beta = 4$.

* `q8a` — the minimum SIR (dB) with $h = 0$ (the box in the corner of Figure D2).
* `q8b` — three `True`/`False` answers, in a list like `[True, False, True]`:
  1. Somewhere on my drive the interference is stronger than the signal (the blue curve goes below 0 dB).
  2. With free-space propagation, my network meets my requirement everywhere on the drive (blue curve never below the
     red line).
  3. Hysteresis lowers the minimum SIR (the $h$ value in the box is below the $h = 0$ value).

In [ ]:
q8a = ...
q8b = ...

In [ ]:
grader.check("q8")

**Question 9 — reflection (1 pt; at least 40 words earns the point).** In 2–4 sentences, inside the triple quotes:
what surprised you in your figures (cluster size, handoffs, hysteresis, or the free-space SIR), or what would you tell a
classmate about choosing $N$ and $h$?

In [ ]:
reflection = ...

In [ ]:
grader.check("q9")

## Connect Homework 1 to the reference radio: UE motion, serving distance, handoff, and co-channel interference (ungraded)

This is not a new, unrelated simulation. The supplied cell below takes arrays that **you already produced in this homework** and passes them into the same pedagogical NR-inspired reference downlink used in lecture. No coding or separate submission is required.

**Causal path:** your drive changes every BS distance → `rx_dbm` changes → `serving_h` chooses the desired link and the co-channel group supplies interference → SINR/BLER/goodput change

**Predict before running:** Find a handoff region or cell edge where desired power falls while aggregate co-channel interference rises.

In [ ]:
import importlib
import numpy as np
from ece512 import radio
radio = importlib.reload(radio)  # pick up course-tool updates in an already-running kernel
_required_radio_api = ('aggregate_interference_dbm', 'evaluate_trace', 'trace_from_sinr', 'plot_trace', 'animate_trace')
_missing_radio_api = [name for name in _required_radio_api if not hasattr(radio, name)]
if _missing_radio_api:
    raise RuntimeError(
        'This notebook is newer than the installed ece512 course tools. '        'Rerun the Setup cell with an updated courseware package, restart the kernel, '        f'and run all cells again. Missing radio API: {_missing_radio_api}'
    )

rr_sample = np.arange(len(positions))
rr_desired = rx_dbm[rr_sample, serving_h]
rr_interference = np.empty(len(positions))
for rr_k, rr_serving in enumerate(serving_h):
    rr_mask = (bs_group == bs_group[rr_serving])
    rr_mask[rr_serving] = False
    rr_interference[rr_k] = radio.aggregate_interference_dbm(rx_dbm[rr_k, rr_mask])
rr_trace = radio.evaluate_trace(traveled, rr_desired, rr_interference, -100,
                                driver_label="distance traveled (m)")
rr_trace = radio.attach_scene(rr_trace, "cellular", ue_xy=positions, gnb_xy=bs_xy,
                              serving_index=serving_h)

radio.plot_trace(rr_trace, title='Homework 1: UE motion, serving distance, handoff, and co-channel interference → reference-radio performance');

### Read your result from left to right

This trace is built from the same `positions`, `rx_dbm`, cell groups, and hysteretic serving-cell decisions used in Figures C and D. A handoff can raise desired power abruptly, while staying too long on the old cell can lower SINR even when the route is smooth.

1. Locate a feature in the **topic-model input** panel. 2. At the same horizontal coordinate, read SINR, BLER, and expected goodput. 3. Confirm the consequence in the received image. If SINR falls, diagnose a physical-link problem; if SINR stays fixed while availability falls, diagnose resource/rate support.

### Animate the same data

The optional cell below synchronizes **the physical situation**, **your homework quantity**, the reference radio's SINR/BLER/goodput response, and the delivered image. All four panels use the same sample at every frame. It is excluded from grading because it is only an explanatory view.

In [ ]:
from IPython.display import HTML
HTML(radio.animate_trace(rr_trace, title='Homework 1: UE motion, serving distance, handoff, and co-channel interference').to_jshtml())

## Optional programming challenge (not graded, no points)
Only if you enjoy programming; skipping it changes nothing. Write the exact worst-case SIR (Figure A2's blue curve)
yourself. For a mobile at the corner `R + 0j` of the home cell (base station at the origin), with the six co-channel base
stations `cells.first_tier_cochannel(0j, i, j, R)` (complex positions in meters),
$$\Lambda = \frac{R^{-\beta}}{\sum_{k=1}^{6} d_k^{-\beta}}, \qquad d_k = |\text{mobile} - \text{base station}_k|,$$
and compare your value for $N = 7$ ($i = 2$, $j = 1$) with Figure A2. Leave the `...` in place if you skip it.

In [ ]:
def my_worst_case_sir_db(i, j, R, beta):
    interferers = cells.first_tier_cochannel(0j, i, j, R)
    ...   # optional: compute and return the SIR in dB

_mine = my_worst_case_sir_db(2, 1, 1.0, 4)
if _mine is None:
    print("Optional challenge not attempted (that's fine).")
else:
    print(f"N = 7, beta = 4: yours {_mine:.2f} dB, Figure A2 {10 * np.log10(pl.worst_case_sir_geometric(2, 1, 1.0, 4)):.2f} dB")

## Submission
Run the cells below to check all public tests and create the zip file. Download the zip file and upload it to Canvas.

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Download the zip file created below and upload it to Canvas.

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)